In [1]:
# ====================================================
# ПРОЕКТ: BIG DATA — PANDAS ПАДАЕТ, DASK РАБОТАЕТ
# ====================================================

print("🚀 ЗАПУСК ПРОЕКТА BIG DATA")
print("="*60)

# Установка библиотек
!pip install dask[dataframe] --quiet

import pandas as pd
import numpy as np
import dask.dataframe as dd
import time
import psutil
import gc

print("✅ Библиотеки установлены")


🚀 ЗАПУСК ПРОЕКТА BIG DATA
✅ Библиотеки установлены


In [ ]:
# ====================================================
# БЛОК 1: PANDAS ПАДАЕТ (2 МЛРД СТРОК)
# ====================================================

print("\n" + "="*60)
print("ТЕСТ 1: PANDAS НА 2 МЛРД СТРОК")
print("="*60)

print("🔄 Попытка создать 2 млрд строк в pandas...")

# Пандас пытается создать 2 млрд строк
df_pandas = pd.DataFrame({
    'store_id': np.random.randint(1, 1001, 2_000_000_000),
    'revenue': np.random.uniform(10, 1000, 2_000_000_000)
})

print("❌ Ошибка: пандас выжил (не должно быть)")


ТЕСТ 1: PANDAS НА 2 МЛРД СТРОК
🔄 Попытка создать 2 млрд строк в pandas...


In [2]:
# ====================================================
# БЛОК 2: DASK ОБРАБАТЫВАЕТ ТЕ ЖЕ 2 МЛРД СТРОК
# ====================================================

print("\n" + "="*60)
print("ТЕСТ 2: DASK НА 2 МЛРД СТРОК")
print("="*60)

n_rows = 2_000_000_000  # ТЕ ЖЕ 2 МЛРД СТРОК!
n_partitions = 100       # Разбиваем на 100 партиций
rows_per_partition = n_rows // n_partitions

print(f"📊 Обработка {n_rows:,} строк через Dask")
print(f"📊 Разбито на {n_partitions} партиций по {rows_per_partition:,} строк")

def gen_partition(i):
    """Генерирует одну партицию данных"""
    np.random.seed(i)
    return pd.DataFrame({
        'store_id': np.random.randint(1, 1001, rows_per_partition),
        'revenue': np.random.uniform(10, 1000, rows_per_partition)
    })

print("🔄 Создание Dask DataFrame...")
ddf = dd.from_map(gen_partition, range(n_partitions))

print("🔄 Агрегация (группировка по store_id)...")
start = time.time()

# Dask обрабатывает данные партиция за партицией
result = ddf.groupby('store_id')['revenue'].sum().compute()

dask_time = time.time() - start

print(f"\n✅ Dask успешно обработал {n_rows:,} строк!")
print(f"⏱️ Время: {dask_time:.2f} сек")
print(f"💾 Память: {psutil.Process().memory_info().rss / 1024**3:.2f} GB")
print(f"\n📊 Общая выручка: ${result.sum():,.2f}")
print(f"📊 Количество магазинов: {len(result)}")


ТЕСТ 2: DASK НА 2 МЛРД СТРОК
📊 Обработка 2,000,000,000 строк через Dask
📊 Разбито на 100 партиций по 20,000,000 строк
🔄 Создание Dask DataFrame...
🔄 Агрегация (группировка по store_id)...

✅ Dask успешно обработал 2,000,000,000 строк!
⏱️ Время: 95.31 сек
💾 Память: 0.28 GB

📊 Общая выручка: $1,010,006,365,133.34
📊 Количество магазинов: 1000


In [3]:
# ====================================================
# БЛОК 3: СРАВНЕНИЕ И ВЫВОД
# ====================================================

print("\n" + "="*60)
print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ")
print("="*60)

print("""
┌─────────────────────────────────────────────────────────────────┐
│                    СРАВНЕНИЕ НА 2 МЛРД СТРОК                    │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   Pandas:   ❌ УПАЛ (не хватило RAM)                           │
│   Dask:     ✅ РАБОТАЕТ (обработал все 2 млрд)                 │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                     ПОЧЕМУ DASK СПРАВИЛСЯ?                      │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   1. Данные разбиты на 100 партиций                             │
│   2. Каждая партиция обрабатывается отдельно                    │
│   3. Результаты агрегируются постепенно                         │
│   4. Не нужно загружать всё в RAM                               │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                              ВЫВОД                              │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   • Pandas требует загрузки ВСЕХ данных в RAM                   │
│   • В крупных проектах объёмы — терабайты → pandas НЕВОЗМОЖЕН   │
│   • Spark/Hadoop работают как Dask — по партициям               │
│                                                                 │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
""")

print("\n✅ ПРОЕКТ ЗАВЕРШЁН!")


СРАВНЕНИЕ РЕЗУЛЬТАТОВ

┌─────────────────────────────────────────────────────────────────┐
│                    СРАВНЕНИЕ НА 2 МЛРД СТРОК                    │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   Pandas:   ❌ УПАЛ (не хватило RAM)                           │
│   Dask:     ✅ РАБОТАЕТ (обработал все 2 млрд)                 │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                     ПОЧЕМУ DASK СПРАВИЛСЯ?                      │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   1. Данные разбиты на 100 партиций                             │
│   2. Каждая партиция обрабатывается отдельно                    │
│   3. Результаты агрегирую